# 03 — ELF Continuous Flow-Matching

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/winstonsmith1897/DantinoX/blob/main/docs/notebooks/03_elf_flow_matching.ipynb)

**ELF (Embedded Language Flow)** is a continuous flow-matching language model that operates in T5 embedding space.

Key concepts:
- `ELFTransformer` — bidirectional transformer with in-context time and CFG control tokens
- Forward process: `z_t = t·x + (1−t)·ε`, interpolates between noise (t=0) and data (t=1)
- **`embed_dim` must match T5-base hidden size = 768** (the flow space lives in T5's embedding space)
- `model_dim` can be set independently (transformer backbone width)
- Classifier-free guidance (CFG) via dedicated control tokens
- Euler ODE sampler at inference

**Runtime**: GPU (T4 or better). T5-base (~850MB) is downloaded on first run and cached.

In [ ]:
# ELF requires JAX + Flax + transformers with Flax backend
!pip install -q git+https://github.com/winstonsmith1897/DantinoX.git#egg=dantinox[all] "transformers[flax]"

In [ ]:
import urllib.request, os
if not os.path.exists('tiny_shakespeare.txt'):
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt',
        'tiny_shakespeare.txt')
    print('Downloaded tiny_shakespeare.txt')

with open('tiny_shakespeare.txt') as f:
    text = f.read()
print(f'Corpus: {len(text):,} characters')

## 1. ELF Configuration

`ELFConfig` has two main dimension parameters:

| Parameter | Description | Constraint |
|---|---|---|
| `embed_dim` | Flow-matching space dimension | **Must equal T5 hidden dim** (768 for T5-base) |
| `bottleneck_dim` | Bottleneck between embed and model space | Any size < embed_dim |
| `model_dim` | Transformer backbone width | Determines `n_heads × head_size` |
| `vocab_size` | Output vocabulary | **Must equal T5 vocab size** (32100 for T5-base) |

In [ ]:
from dantinox.core.config import ELFConfig
from dantinox.core.elf import ELFTransformer
from flax import nnx
import jax

# embed_dim=768 matches T5-base hidden size
# model_dim=256 is the transformer backbone (can be smaller than embed_dim)
config = ELFConfig(
    embed_dim      = 768,   # T5-base hidden size — MUST match T5 output dim
    bottleneck_dim = 128,   # projects 768 → 128 → model_dim
    model_dim      = 256,   # transformer hidden dim
    n_heads        = 4,
    head_size      = 64,    # model_dim = n_heads × head_size = 4×64 = 256 ✓
    num_blocks     = 4,
    vocab_size     = 32100, # T5 tokenizer vocabulary size
    max_seq_len    = 128,
    dropout        = 0.0,
    gradient_checkpointing = False,
)

model  = ELFTransformer(config, rngs=nnx.Rngs(42))
n_params = sum(x.size for x in jax.tree_util.tree_leaves(nnx.state(model, nnx.Param)))
print(f'ELFTransformer — {n_params / 1e6:.2f}M parameters')

## 2. Forward Process

ELF's forward process interpolates linearly between Gaussian noise and clean T5 embeddings:

```
z_t = t·x + (1−t)·ε     ε ~ N(0, I),  t ∈ [0, 1]
```

- `t = 0`: pure noise  
- `t = 1`: clean embedding

In [ ]:
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

rng = jax.random.PRNGKey(0)
eps = jax.random.normal(rng, (768,))   # noise vector
x   = jnp.ones(768) * 0.5             # mock clean embedding

ts   = np.linspace(0, 1, 6)
fig, axes = plt.subplots(1, 6, figsize=(14, 2.5))
for ax, t in zip(axes, ts):
    z_t = t * x + (1 - t) * eps
    ax.hist(np.array(z_t), bins=25, color='steelblue', alpha=0.8)
    ax.set_title(f't = {t:.1f}')
    ax.set_xlim(-3.5, 3.5)
    ax.set_xlabel('z_t')
axes[0].set_ylabel('count')
plt.suptitle('ELF forward process: z_t = t·x + (1−t)·ε', y=1.02)
plt.tight_layout(); plt.show()

## 3. Time Schedule

Training uses a **logit-normal** schedule: concentrates samples in the hard denoising regime near `t ≈ 0.2`.

In [ ]:
from dantinox.core.diffusion import logit_normal_schedule

# logit_normal_schedule(n) returns n+1 time-points: [0, interiors, 1]
train_ts = logit_normal_schedule(64, pmean=-1.5, pstd=0.8)
infer_ts = jnp.linspace(0.0, 1.0, 65)

print(f'Train schedule: {len(train_ts)} steps, shape={train_ts.shape}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
ax1.hist(np.array(train_ts[1:-1]), bins=30, color='coral', alpha=0.8)
ax1.set_xlabel('t'); ax1.set_title('Logit-normal training schedule')
ax2.plot(np.array(infer_ts), marker='.', color='steelblue')
ax2.set_xlabel('step'); ax2.set_ylabel('t'); ax2.set_title('Uniform inference schedule')
plt.tight_layout(); plt.show()

## 4. Training

ELF uses T5-base as a **frozen text encoder**. The `Trainer` automatically:
1. Encodes each batch through T5-base → 768-dim embeddings
2. Normalises the embeddings (channel-wise mean/std, computed once before training)
3. Runs the ELF denoising loss + decoder CE loss

> ⚠️ T5-base (~850 MB) is downloaded on first run. `tokenizer_type` is forced to `"t5"` automatically.

In [ ]:
import dantinox as dx

elf_cfg = dx.ELFConfig(
    embed_dim      = 768,    # must match T5-base
    bottleneck_dim = 128,
    model_dim      = 256,
    n_heads        = 4,
    head_size      = 64,
    num_blocks     = 4,
    vocab_size     = 32100,  # T5 vocabulary size
    max_seq_len    = 128,
    elf_n_steps    = 32,
    elf_cfg_scale  = 1.5,
)

paradigm  = dx.ContinuousParadigm(elf_cfg)
train_cfg = dx.TrainingConfig(lr=1e-3, epochs=1, batch_size=8)

run_dir = dx.Trainer(paradigm, train_cfg).fit('tiny_shakespeare.txt')
print('Checkpoint saved to:', run_dir)

## 5. Generation

Generation denoises from Gaussian noise to clean embeddings via an Euler ODE, then decodes:

```
z_0 ~ N(0, I)  →  ODE steps (t: 0 → 1)  →  argmax(W_e^T z_1)  →  tokens
```

In [ ]:
from dantinox.core.generation import elf_generate

elf_model = dx.load(run_dir, paradigm=paradigm)

tokens = elf_generate(
    elf_model, gen_len=32, batch_size=4,
    n_steps=32, cfg_scale=elf_cfg.elf_cfg_scale, seed=42,
)
print('Generated token IDs shape:', tokens.shape)
print('Sample:', tokens[0, :20].tolist())

## 6. CFG Guidance Scale Ablation

Classifier-Free Guidance scale `w` controls diversity vs. focus:
- `w = 1.0`: no guidance
- `w = 1.5`: mild guidance (recommended default)
- `w ≥ 3.0`: strong guidance (more focused, may reduce diversity)

In [ ]:
for w in (1.0, 1.5, 2.0, 3.0, 5.0):
    toks = elf_generate(elf_model, gen_len=32, batch_size=1,
                        n_steps=32, cfg_scale=w, seed=42)
    unique = len(set(toks[0].tolist()))
    print(f'cfg_scale={w:.1f}  unique_tokens={unique:3d}  sample={toks[0, :12].tolist()}')

## 7. Model Size Ablation

`embed_dim` is fixed at 768 (T5-base), but `model_dim` and `num_blocks` can be varied freely.

In [ ]:
size_variants = [
    ('Tiny',   dx.ELFConfig(embed_dim=768, bottleneck_dim=64,  model_dim=64,
                             n_heads=2, head_size=32, num_blocks=2,
                             vocab_size=32100, max_seq_len=64)),
    ('Small',  dx.ELFConfig(embed_dim=768, bottleneck_dim=128, model_dim=256,
                             n_heads=4, head_size=64, num_blocks=4,
                             vocab_size=32100, max_seq_len=128)),
    ('Medium', dx.ELFConfig(embed_dim=768, bottleneck_dim=256, model_dim=512,
                             n_heads=8, head_size=64, num_blocks=8,
                             vocab_size=32100, max_seq_len=256)),
]

for name, cfg in size_variants:
    m = ELFTransformer(cfg, rngs=nnx.Rngs(0))
    n = sum(x.size for x in jax.tree_util.tree_leaves(nnx.state(m, nnx.Param)))
    print(f'{name:8s}  model_dim={cfg.model_dim:4d}  blocks={cfg.num_blocks:2d}  {n/1e6:.2f}M params')

## 8. One-liner API

In [ ]:
rd = dx.fit(
    'continuous', 'tiny_shakespeare.txt',
    embed_dim=768, bottleneck_dim=128, model_dim=256,
    n_heads=4, head_size=64, num_blocks=4,
    vocab_size=32100, max_seq_len=128,
    elf_n_steps=32, elf_cfg_scale=1.5,
    lr=1e-3, epochs=1, batch_size=8,
)
print('ELF run dir:', rd)